<font color='rainbow' size=6pt> Supervised Machine Learning: Match Predictor

In [ ]:
#set up environment
!python set_up.py

import os
pip_upgrade = os.path.join("venv", "Scripts", "python.exe") if os.name == "nt" else os.path.join("venv", "bin", "python")
!{pip_upgrade} -m pip install --upgrade pip

!pip install -r https://raw.githubusercontent.com/Keegan-McCullough/Soccer_Match_Predictor/main/requirements.txt

In [ ]:
import kagglehub
import pandas as pd
import os
import sqlite3

# Download latest version
epl_path = kagglehub.dataset_download("evangower/english-premier-league-standings")

epl_dfs = {}
# Load EPL CSVs into a dictionary of DataFrames
for f in os.listdir(epl_path):
    if f.endswith(".csv"):
        epl_dfs[f] = pd.read_csv(os.path.join(epl_path, f))


# Download Europe soccer SQLite database
europe_path = kagglehub.dataset_download("hugomathien/soccer")

# Find the file
db_file = [f for f in os.listdir(europe_path) if f.endswith(".sqlite")][0]
db_path = os.path.join(europe_path, db_file)

# Connect and list all tables
conn = sqlite3.connect(db_path)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
# Load each table into a dictionary of DataFrames
europe_dfs = {}
for table in tables["name"]:
    europe_dfs[table] = pd.read_sql(f"SELECT * FROM {table}", conn)
conn.close()

In [ ]:
# EPL data structure overview
print("EPL datasets:", list(epl_dfs.keys()))
epl_rankings = epl_dfs['pl-tables-1993-2024.csv']
print(epl_rankings)

In [ ]:
# Europe DB tables overview
print("\nEurope DB tables:", tables["name"].tolist())

# Get all the data for only the englis teams
english_teams = ['Arsenal','Aston Villa', 'Blackburn Rovers', 'Bolton Wanderers', 
                 'Chelsea', 'Everton', 'Liverpool', 'Manchester City', 'Manchester United',
                 'Middlesbrough', 'Newcastle United', 'Queens Park Rangers', 'Southampton',
                 'Tottenham Hotspur', 'West Ham United', 'Burnley', 'Swansea', 'Wolverhampton Wanderers',
                 'Cardiff City', 'Hull City', 'Stoke City', 'Norwich City', 'West Bromwich Albion',
                 'Blackpool', 'Crystal Palace', 'Watford', 'Leicester City', 'Sunderland', 'Birmingham City',
                 'Fulham', 'Wigan Athletic', 'Swansea City', 'Reading', 'Bournemouth']

df_teams = europe_dfs["Team"]
team_map = {}

for index, team in df_teams.iterrows():
    if team['team_long_name'] in english_teams:
        team_map[team['team_api_id']] = team['team_long_name']

# map ids to name for readability 
def map_ids(df, team_map, name):
    df[name] = df[name].map(team_map)
    return df

# Collect only the data we want ie discrete values
df_attr = europe_dfs['Team_Attributes'][['team_api_id', 'date', 'buildUpPlaySpeed', 'buildUpPlayPassing',
                                         'chanceCreationPassing', 'chanceCreationCrossing', 'chanceCreationShooting',
                                         'defencePressure', 'defenceAggression', 'defenceTeamWidth' ]]

df_attr = df_attr[df_attr['team_api_id'].isin(team_map.keys())]
df_attr = map_ids(df_attr, team_map, 'team_api_id')

print(team_map)

df_match = europe_dfs['Match'][['league_id', 'season', 'date', 'home_team_api_id', 'away_team_api_id', 'home_team_goal',
                                'away_team_goal', 'B365H', 'B365D', 'B365A']]

df_match = map_ids(df_match, team_map, 'home_team_api_id')
df_match = map_ids(df_match, team_map, 'away_team_api_id')

epl = 1729
df_match = df_match[df_match['league_id'] == epl]
df_match = df_match[df_match['season'] != '2008/2009']
df_match = df_match[df_match['season'] != '2009/2010']

df_attr['date'] = pd.to_datetime(df_attr['date'])
df_match['date'] = pd.to_datetime(df_match['date'])

df_attr = df_attr.sort_values('date')
df_match = df_match.sort_values('date')

print(df_match.head())
#print(df_attr.head())


In [ ]:
# Feature engineering and df connections

# Home team attributes
df_match = pd.merge_asof(
    df_match,
    df_attr,
    left_on='date',
    right_on='date',
    left_by='home_team_api_id',
    right_by='team_api_id',
    direction='backward'
)

df_match = df_match.rename(columns=lambda x: f"home_{x}" if x not in ['league_id','season','date','home_team_api_id','away_team_api_id','home_team_goal','away_team_goal','B365H','B365D','B365A'] else x)

# Repeat for away team
df_match = pd.merge_asof(
    df_match,
    df_attr,
    left_on='date',
    right_on='date',
    left_by='away_team_api_id',
    right_by='team_api_id',
    direction='backward'
)

df_match = df_match.rename(columns=lambda x: f"away_{x}" if 'home_' not in x and x not in ['league_id','season','date','home_team_api_id','away_team_api_id','home_team_goal','away_team_goal','B365H','B365D','B365A'] else x)

